In [1]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
  'daducci_amico',
  'yangjasp_optimall',
  'anybotics_kindr',
  'scikit-fuzzy_scikit-fuzzy',
  'mhhennig_hs2',
  'bids-apps_rshrf',
  'juliastats_lasso.jl',
  'dcc-lab_pyhardwarelibrary',
  'cmillion_gphoton',
  'aim-uofa_adelaidet'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('eabbott9_df_commits.csv')
df.head(1)

,sha1,project
0,00534996cc67dc37a0406ba20f8d16a68fa3ff79,daducci_amico


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():

    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('eabbott9_df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|          | 7/652 [00:07<10:56,  1.02s/it]

Got Errors {'231f31b8ddcbb2e79fd1466af9a612dfcfb459c9': 'Key 231f31b8ddcbb2e79fd1466af9a612dfcfb459c9 not found in /da5_fast/All.sha1c/commit_35.tch'}


 25%|██▌       | 164/652 [02:46<08:15,  1.02s/it]

Got Errors {'86db8a5ea3b6cd902bf2b406a61763eb6bb063a7': 'Key 86db8a5ea3b6cd902bf2b406a61763eb6bb063a7 not found in /da5_fast/All.sha1c/commit_6.tch'}


 31%|███       | 200/652 [03:23<07:38,  1.02s/it]

...

Got Errors {'2c5ab9c369d9119581869c1f1a1016582b1daa40': 'Key 2c5ab9c369d9119581869c1f1a1016582b1daa40 not found in /da5_fast/All.sha1c/commit_44.tch'}


 82%|████████▏ | 534/652 [09:02<01:59,  1.02s/it]

Got Errors {'586de90be1847ded96d7c50aef291b0ba5099ad9': 'Key 586de90be1847ded96d7c50aef291b0ba5099ad9 not found in /da5_fast/All.sha1c/commit_88.tch'}


100%|██████████| 652/652 [11:02<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00534996cc67dc37a0406ba20f8d16a68fa3ff79,21b7f5309dc04f392df524bd61b459eab3a0817f,[d65239cc8bc5fe7da12ca365aaa552792b0bc0ce],nightwnvol <notte_94@hotmail.it>,1656677387,+0200,nightwnvol <notte_94@hotmail.it>,1656677387,+0200,refactor: rename methods and variables\n,00534996cc67dc37a0406ba20f8d16a68fa3ff79,daducci_amico
1,00c6ac9579f55819e5e800aba08f9cf18b314c0a,174834197983965595f4ce17e9b58a916be47070,[09cfbdda847232fd5b85f1cf00b4c5b8c711086f],Alessandro Daducci <alessandro.daducci@univr.it>,1638532643,+0100,Alessandro Daducci <alessandro.daducci@univr.it>,1638532643,+0100,Install information are stored (and taken from...,00c6ac9579f55819e5e800aba08f9cf18b314c0a,daducci_amico


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00534996cc67dc37a0406ba20f8d16a68fa3ff79', 'tree': '21b7f5309dc04f392df524bd61b459eab3a0817f', 'parent': ['d65239cc8bc5fe7da12ca365aaa552792b0bc0ce'], 'author': 'nightwnvol <notte_94@hotmail.it>', 'author_time': 1656677387, 'author_tz': '+0200', 'committer': 'nightwnvol <notte_94@hotmail.it>', 'committer_time': 1656677387, 'committer_tz': '+0200', 'message': 'refactor: rename methods and variables\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
new_columns = ['project', 'commit', 'author', 'time', 'message']
dfinf = df_commit_data.rename(columns={
    'author_time': 'time',
})[new_columns]


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,daducci_amico,00534996cc67dc37a0406ba20f8d16a68fa3ff79,nightwnvol <notte_94@hotmail.it>,1656677387,refactor: rename methods and variables\n


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='eabbott9'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Answers

### Table of the number of stars, number of forks, and the last commit date for each project


In [8]:
# defines project dataframes with information from GitHub
github_cols = ['Project', 'Stars', 'Forks', 'LastCommitDate']
github_stats = [
    ('daducci_amico', 130, 67, '12/17/2025'),
    ('yangjasp_optimall', 7, 2, '4/3/2026'),
    ('anybotics_kindr', 617, 207, '2/14/2025'),
    ('scikit-fuzzy_scikit-fuzzy', 877, 292, '8/25/2024'),
    ('mhhennig_hs2', 31, 19, '1/24/2025'),
    ('bids-apps_rshrf', 42, 15, '7/23/2026'),
    ('juliastats_lasso.jl', 147, 34, '7/28/2026'),
    ('dcc-lab_pyhardwarelibrary', 12, 3, '8/12/2026'),
    ('cmillion_gphoton', 21, 10, '2/13/2025'),
    ('aim-uofa_adelaidet', 3500, 653, '8/22/2024'),
]
project_level_df = pd.DataFrame(github_stats, columns=github_cols)

print(project_level_df.to_markdown(index=False))
print()

| Project                   |   Stars |   Forks | LastCommitDate   |
|:--------------------------|--------:|--------:|:-----------------|
| daducci_amico             |     130 |      67 | 12/17/2025       |
| yangjasp_optimall         |       7 |       2 | 4/3/2026         |
| anybotics_kindr           |     617 |     207 | 2/14/2025        |
| scikit-fuzzy_scikit-fuzzy |     877 |     292 | 8/25/2024        |
| mhhennig_hs2              |      31 |      19 | 1/24/2025        |
| bids-apps_rshrf           |      42 |      15 | 7/23/2026        |
| juliastats_lasso.jl       |     147 |      34 | 7/28/2026        |
| dcc-lab_pyhardwarelibrary |      12 |       3 | 8/12/2026        |
| cmillion_gphoton          |      21 |      10 | 2/13/2025        |
| aim-uofa_adelaidet        |    3500 |     653 | 8/22/2024        |



| Project                   |   Stars |   Forks | LastCommitDate   |
|:--------------------------|--------:|--------:|:-----------------|
| daducci_amico             |     130 |      67 | 12/17/2025       |
| yangjasp_optimall         |       7 |       2 | 4/3/2026         |
| anybotics_kindr           |     617 |     207 | 2/14/2025        |
| scikit-fuzzy_scikit-fuzzy |     877 |     292 | 8/25/2024        |
| mhhennig_hs2              |      31 |      19 | 1/24/2025        |
| bids-apps_rshrf           |      42 |      15 | 7/23/2026        |
| juliastats_lasso.jl       |     147 |      34 | 7/28/2026        |
| dcc-lab_pyhardwarelibrary |      12 |       3 | 8/12/2026        |
| cmillion_gphoton          |      21 |      10 | 2/13/2025        |
| aim-uofa_adelaidet        |    3500 |     653 | 8/22/2024        |


### Table of the number of commits, the number of authors, and max and min time for each project

In [9]:
# generates partial markdown table for project, author, max_time, and min_time
num_commits = []
num_authors = []
max_time = []
min_time = []
for project in projects:
    project_df = dfinf[dfinf['project'] == project]
    num_commits.append(len(project_df))
    num_authors.append(project_df['author'].nunique())
    max_time.append(project_df['time'].max())
    min_time.append(project_df['time'].min())
project_level_df['NumCommits'] = num_commits
project_level_df['NumAuthors'] = num_authors
project_level_df['MaxTime'] = max_time
project_level_df['MinTime'] = min_time

project_level_df.to_csv("eabbott9_project_stats.csv", index=False)

print(project_level_df.to_markdown(index=False))
print()

| Project                   |   Stars |   Forks | LastCommitDate   |   NumCommits |   NumAuthors |    MaxTime |    MinTime |
|:--------------------------|--------:|--------:|:-----------------|-------------:|-------------:|-----------:|-----------:|
| daducci_amico             |     130 |      67 | 12/17/2025       |          618 |           46 | 1768769157 | 1412688947 |
| yangjasp_optimall         |       7 |       2 | 4/3/2026         |          564 |            3 | 1775254108 | 1600887255 |
| anybotics_kindr           |     617 |     207 | 2/14/2025        |          864 |           65 | 1741698942 | 1381244746 |
| scikit-fuzzy_scikit-fuzzy |     877 |     292 | 8/25/2024        |          673 |           68 | 1731068774 | 1363667243 |
| mhhennig_hs2              |      31 |      19 | 1/24/2025        |          614 |           30 | 1737717436 | 1501257793 |
| bids-apps_rshrf           |      42 |      15 | 7/23/2026        |          553 |           31 | 1775331973 | 1468998858 |


| Project                   |   Stars |   Forks | LastCommitDate   |   NumCommits |   NumAuthors |    MaxTime |    MinTime |
|:--------------------------|--------:|--------:|:-----------------|-------------:|-------------:|-----------:|-----------:|
| daducci_amico             |     130 |      67 | 12/17/2025       |          618 |           46 | 1768769157 | 1412688947 |
| yangjasp_optimall         |       7 |       2 | 4/3/2026         |          564 |            3 | 1775254108 | 1600887255 |
| anybotics_kindr           |     617 |     207 | 2/14/2025        |          864 |           65 | 1741698942 | 1381244746 |
| scikit-fuzzy_scikit-fuzzy |     877 |     292 | 8/25/2024        |          673 |           68 | 1731068774 | 1363667243 |
| mhhennig_hs2              |      31 |      19 | 1/24/2025        |          614 |           30 | 1737717436 | 1501257793 |
| bids-apps_rshrf           |      42 |      15 | 7/23/2026        |          553 |           31 | 1775331973 | 1468998858 |
| juliastats_lasso.jl       |     147 |      34 | 7/28/2026        |          368 |           28 | 1762100330 | 1396912911 |
| dcc-lab_pyhardwarelibrary |      12 |       3 | 8/12/2026        |          724 |           16 | 1758678540 | 1531878900 |
| cmillion_gphoton          |      21 |      10 | 2/13/2025        |          949 |           12 | 1762296960 | 1394033282 |
| aim-uofa_adelaidet        |    3500 |     653 | 8/22/2024        |          573 |           55 | 1725492812 | 1579751240 |
